# 02 · Fallback voice — Kokoro-82M (GPU)

**Kernel:** `JARVIS - kokoro` (venv `C:\jarvis-venvs\kokoro`, Python 3.11)

A working British voice while the GPT-SoVITS clone is trained, and a latency/VRAM baseline to compare it against.
No streaming path — this is the fallback. For clean VRAM numbers: **Restart kernel → Run All**.

In [ ]:
import os, sys, time, json, statistics
from pathlib import Path

os.environ.setdefault("HF_HOME", r"C:\jarvis-models\hf")

OUT = Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)

REPO_ID = "hexgrad/Kokoro-82M"
VOICE = "bm_george"
TEXT = "Good morning, Sir."
SAMPLE_RATE = 24000

assert sys.version_info[:2] in [(3, 11), (3, 12)], sys.version
print(sys.executable)

In [ ]:
# --- VRAM monitor: create BEFORE anything touches CUDA ---------------------------------
# process MB : this process's dedicated memory on the NVIDIA adapter, from the Windows
#              "GPU Process Memory" counter (what Task Manager shows). NVML cannot see
#              per-process usage under WDDM; this counter can. Headline figure.
# device delta MB : NVML total used minus the baseline. Includes other apps - cross-check.
import ctypes, threading
from ctypes import wintypes
import pynvml

class _FmtValue(ctypes.Structure):
    _fields_ = [("CStatus", wintypes.DWORD), ("largeValue", ctypes.c_longlong)]

class _FmtItem(ctypes.Structure):
    _fields_ = [("szName", wintypes.LPWSTR), ("FmtValue", _FmtValue)]

class PdhWildcard:
    """Every instance of a wildcard Windows performance counter, as {instance: bytes}."""
    def __init__(self, path):
        self.pdh, self.path = ctypes.WinDLL("pdh"), path
        self.query, self.counter = wintypes.HANDLE(), wintypes.HANDLE()
        self.reopen()

    def reopen(self):
        if self.query.value:
            self.pdh.PdhCloseQuery(self.query)
        self.query, self.counter = wintypes.HANDLE(), wintypes.HANDLE()
        if self.pdh.PdhOpenQueryW(None, None, ctypes.byref(self.query)) != 0:
            raise OSError("PdhOpenQueryW failed")
        if self.pdh.PdhAddEnglishCounterW(self.query, self.path, None, ctypes.byref(self.counter)) != 0:
            raise OSError(f"cannot open counter {self.path}")

    def read(self):
        if self.pdh.PdhCollectQueryData(self.query) != 0:
            return {}
        size, count = wintypes.DWORD(0), wintypes.DWORD(0)
        self.pdh.PdhGetFormattedCounterArrayW(self.counter, 0x400, ctypes.byref(size), ctypes.byref(count), None)
        if size.value == 0:
            return {}
        buf = (ctypes.c_byte * size.value)()
        if self.pdh.PdhGetFormattedCounterArrayW(self.counter, 0x400, ctypes.byref(size), ctypes.byref(count), buf) != 0:
            return {}
        items = ctypes.cast(buf, ctypes.POINTER(_FmtItem * count.value)).contents
        return {i.szName: i.FmtValue.largeValue for i in items if i.szName}

class VramMonitor:
    def __init__(self, interval=0.05):
        pynvml.nvmlInit()
        self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        self.baseline = pynvml.nvmlDeviceGetMemoryInfo(self.handle).used
        # Hybrid laptop: pick the adapter LUID whose usage matches NVML, i.e. the RTX 4060.
        adapters = PdhWildcard(r"\GPU Adapter Memory(*)\Dedicated Usage").read()
        nvidia = min(adapters, key=lambda name: abs(adapters[name] - self.baseline))
        self.luid = nvidia.split("_phys")[0]
        self.prefix = f"pid_{os.getpid()}_{self.luid}"
        self.procs = PdhWildcard(r"\GPU Process Memory(*)\Dedicated Usage")
        self.peak_proc = self.peak_dev = 0
        self.stages, self._last_reopen = [], time.perf_counter()
        self._lock, self._stop = threading.Lock(), threading.Event()
        threading.Thread(target=self._run, args=(interval,), daemon=True).start()
        print(f"{pynvml.nvmlDeviceGetName(self.handle)} | adapter {self.luid} | "
              f"baseline used by other apps: {self.baseline / 2**20:.0f} MB")

    def _run(self, interval):
        while not self._stop.is_set():
            self.sample()
            time.sleep(interval)

    def sample(self):
        mine = [v for n, v in self.procs.read().items() if n.startswith(self.prefix)]
        if not mine and time.perf_counter() - self._last_reopen > 1:
            self.procs.reopen()  # our instance only appears once CUDA creates a context
            self._last_reopen = time.perf_counter()
        proc = max(mine, default=0)
        dev = pynvml.nvmlDeviceGetMemoryInfo(self.handle).used - self.baseline
        with self._lock:
            self.peak_proc, self.peak_dev = max(self.peak_proc, proc), max(self.peak_dev, dev)
        return proc, dev

    def stage(self, label):
        proc, dev = self.sample()
        entry = {"stage": label, "process_now_mb": round(proc / 2**20), "process_peak_mb": round(self.peak_proc / 2**20),
                 "device_delta_now_mb": round(dev / 2**20), "device_delta_peak_mb": round(self.peak_dev / 2**20)}
        self.stages.append(entry)
        print(f"VRAM [{label}] now {entry['process_now_mb']} MB, peak {entry['process_peak_mb']} MB "
              f"(device delta now {entry['device_delta_now_mb']}, peak {entry['device_delta_peak_mb']})")
        return entry

    def summary(self):
        self.sample()
        return {"peak_process_mb": round(self.peak_proc / 2**20), "peak_device_delta_mb": round(self.peak_dev / 2**20),
                "baseline_other_apps_mb": round(self.baseline / 2**20), "stages": self.stages}

vram = VramMonitor()

In [ ]:
from huggingface_hub import list_repo_files
voices = sorted(Path(f).stem for f in list_repo_files(REPO_ID) if f.startswith("voices/b"))
print("British voices:", voices)
assert VOICE in voices, f"{VOICE} is not in the repo"

In [ ]:
t = time.perf_counter()
import numpy as np, sounddevice as sd, soundfile as sf, torch, kokoro
from kokoro import KPipeline
import_ms = (time.perf_counter() - t) * 1000

t = time.perf_counter()
pipeline = KPipeline(lang_code="b", repo_id=REPO_ID, device="cuda")   # "b" = British English
pipeline.load_voice(VOICE)
load_ms = (time.perf_counter() - t) * 1000
print(f"kokoro {kokoro.__version__} | torch {torch.__version__} | CUDA {torch.cuda.is_available()} | "
      f"imports {import_ms:.0f} ms | model+voice load {load_ms:.0f} ms")
vram.stage("after_load")

In [ ]:
def speak_timed(text, voice=VOICE, play=True):
    """Synthesise and play. Time-to-first-audio = call -> playback started."""
    t0 = time.perf_counter()
    chunks, first_ms, play_ms = [], None, None
    for result in pipeline(text, voice=voice, speed=1.0):
        chunks.append(result.audio.detach().cpu().numpy())
        if first_ms is None:
            first_ms = (time.perf_counter() - t0) * 1000
            if play:
                sd.play(chunks[0], SAMPLE_RATE)
                play_ms = (time.perf_counter() - t0) * 1000
    synth_ms = (time.perf_counter() - t0) * 1000
    if play:
        sd.wait()
        if len(chunks) > 1:                     # fallback keeps it simple: the rest plays afterwards
            sd.play(np.concatenate(chunks[1:]), SAMPLE_RATE); sd.wait()
    audio = np.concatenate(chunks)
    return audio, {"ttfa_ms": round(play_ms or first_ms), "synth_total_ms": round(synth_ms),
                   "audio_s": round(len(audio) / SAMPLE_RATE, 2)}

_, cold = speak_timed("Systems are online.", play=False)
print("cold (first inference, not played):", cold)
speak_timed(TEXT, play=False)                                 # first pass over this sentence's G2P path
warm_runs = [speak_timed(TEXT, play=False)[1] for _ in range(3)]
audio, spoken = speak_timed(TEXT)
warm = {"ttfa_ms": round(statistics.median(r["ttfa_ms"] for r in warm_runs)),
        "runs_ms": [r["ttfa_ms"] for r in warm_runs], "spoken_ttfa_ms": spoken["ttfa_ms"],
        "synth_total_ms": spoken["synth_total_ms"], "audio_s": spoken["audio_s"]}
print(f"warm {TEXT!r} (median of 3):", warm)
sf.write(OUT / "kokoro_good_morning.wav", audio, SAMPLE_RATE)
vram.stage("after_speak")

Optional: audition every British male voice (saved to `outputs/`, not played).

In [ ]:
for v in [v for v in voices if v.startswith("bm_")]:
    pipeline.load_voice(v)                                    # first use downloads the voice pack: keep it out of the timing
    speak_timed("Sir, your class begins in thirty minutes.", voice=v, play=False)
    a, stats = speak_timed("Sir, your class begins in thirty minutes.", voice=v, play=False)
    sf.write(OUT / f"kokoro_{v}.wav", a, SAMPLE_RATE)
    print(v, stats)

In [ ]:
summary = {"voice": VOICE, "text": TEXT, "kokoro": kokoro.__version__, "torch": torch.__version__,
           "load_ms": round(load_ms), "cold": cold, "warm": warm, "vram": vram.summary(),
           "torch_max_reserved_mb": round(torch.cuda.max_memory_reserved() / 2**20)}
(OUT / "kokoro_results.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"TTFA {warm['ttfa_ms']} ms | peak VRAM {summary['vram']['peak_process_mb']} MB process "
      f"/ {summary['vram']['peak_device_delta_mb']} MB device delta | torch reserved {summary['torch_max_reserved_mb']} MB")